In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import os

sc.set_figure_params(dpi=100, color_map='viridis_r', transparent=False, frameon=False)
sc.settings.verbosity = 1
sc.logging.print_header()

FIG_DIR = 'figures/4_Sub_clustering'
os.makedirs(FIG_DIR, exist_ok=True)
sc.settings.figdir = FIG_DIR

%config InlineBackend.figure_format = 'retina'

## 加载数据

In [ ]:
adata = sc.read_h5ad('write/0_integrate_two_datasets/adata2.h5ad')
adata.uns['log1p']['base'] = None
print(adata.obs['cell_type'].value_counts())
adata

## 1. Myeloid 亚群分析

Macrophage + Microglia + Monocyte = GBM 免疫微环境的核心

In [ ]:
# 提取髓系细胞
myeloid_types = ['Macrophage', 'Microglia', 'Monocyte', 'Neutrophil']
adata_mye = adata[adata.obs['cell_type'].isin(myeloid_types)].copy()
print(f"Myeloid: {adata_mye.n_obs} cells")
adata_mye.obs['cell_type'].value_counts()

In [ ]:
# 重新做 HVG + PCA + Harmony + neighbors + UMAP
sc.pp.highly_variable_genes(adata_mye, n_top_genes=3000, batch_key=None)
sc.tl.pca(adata_mye, n_comps=50, use_highly_variable=True)

import scanpy.external as sce
sce.pp.harmony_integrate(adata_mye, key='sample', basis='X_pca', adjusted_basis='X_pca_harmony')

sc.pp.neighbors(adata_mye, n_pcs=30, use_rep='X_pca_harmony')
sc.tl.umap(adata_mye, min_dist=0.3)
print("HVG → PCA → Harmony → UMAP done")

In [ ]:
# leiden 亚群
sc.tl.leiden(adata_mye, key_added='leiden_sub', resolution=0.8)
sc.pl.umap(adata_mye, color=['cell_type', 'leiden_sub'], ncols=2, legend_loc='on data',
           save='_myeloid_overview.png')

In [ ]:
# 髓系 marker dotplots — 分三类: identity / immunoreg / functional

identity_markers = {
    "Microglia": ["P2RY12", "TMEM119", "CX3CR1", "SALL1", "MEF2C"],
    "Macrophage": ["APOE", "APOC1", "GPNMB", "CTSD", "CD68", "AIF1", "LST1"],
    "Monocyte": ["VCAN", "FCN1", "CCR2", "CD14", "S100A8", "S100A9", "LYZ"],
    "cDC": ["FCER1A", "CLEC10A", "CD1C", "CD1E", "HLA-DPB1", "CD74"],
    "Neutrophil": ["S100A8", "S100A9", "FCGR3B", "CSF3R"],
}

immunoreg_markers = {
    "Systemic-Inflammatory": ["IL1B", "IL1A", "CCL2", "TNF", "OSM", "CXCL8", "CD83"],
    "Mg-Inflammatory": ["CXCR4", "CXCL12", "CCL3", "CCL4", "CX3CR1", "RHOB", "JUN", "KLF2", "EGR1", "PDK4", "P2RY13"],
    "Complement-IS": ["C1QA", "C1QB", "C1QC", "C3", "VSIG4", "CD163", "SELENOP"],
    "Scavenger-IS": ["MRC1", "MSR1", "CD163", "LYVE1", "COLEC12", "STAB1", "NRP1", "RNASE1", "CTSB"],
}

functional_markers = {
    "Hypoxia": ["ENO1", "HK2", "P4HB", "SLC2A1", "LDHA"],
    "IFN-resp": ["MX1", "IFI6", "STAT1", "IFIT1", "ISG15"],
    "HS-UPR": ["HSPA1A", "DNAJB1", "IER5", "HSPA1B", "HSP90AA1"],
    "G2-M": ["TOP2A", "CENPF", "ASPM", "CCNB1", "CDC20", "PLK1", "UBE2C", "MKI67"],
    "G1-S": ["PCNA", "CLSPN", "TYMS", "MCM7", "TK1", "MCM4", "UHRF1", "GINS2"],
}

sc.tl.dendrogram(adata_mye, groupby='leiden_sub')
sc.pl.dotplot(adata_mye, identity_markers, groupby='leiden_sub', dendrogram=True,
              save='_myeloid_identity_markers.png')
sc.pl.dotplot(adata_mye, immunoreg_markers, groupby='leiden_sub', dendrogram=True,
              save='_myeloid_immunoreg_markers.png')
sc.pl.dotplot(adata_mye, functional_markers, groupby='leiden_sub', dendrogram=True,
              save='_myeloid_functional_markers.png')

In [ ]:
# 找各亚群 DEG
sc.tl.rank_genes_groups(adata_mye, groupby='leiden_sub', method='wilcoxon', n_genes=10)
sc.pl.rank_genes_groups_dotplot(adata_mye, n_genes=5, save='_myeloid_deg.png')

In [ ]:
# === Marker-based scoring: 对三组 marker 分别打分, 自动生成 cluster 注释建议 ===
all_marker_sets = {**identity_markers, **immunoreg_markers, **functional_markers}

# 逐 module 打分
for module_name, genes in all_marker_sets.items():
    genes_use = [g for g in genes if g in adata_mye.var_names]
    sc.tl.score_genes(adata_mye, gene_list=genes_use, score_name=f'score_{module_name}', use_raw=False)

score_cols = [f'score_{m}' for m in all_marker_sets]
cluster_score_mean = adata_mye.obs.groupby('leiden_sub')[score_cols].mean().round(3)

# 自动生成三组来源的 best-hit 标签
identity_cols = [f'score_{m}' for m in identity_markers]
immunoreg_cols = [f'score_{m}' for m in immunoreg_markers]
functional_cols = [f'score_{m}' for m in functional_markers]

auto_labels = pd.DataFrame(index=cluster_score_mean.index)
auto_labels['best_identity'] = cluster_score_mean[identity_cols].idxmax(axis=1).str.replace('score_', '')
auto_labels['best_immunoreg'] = cluster_score_mean[immunoreg_cols].idxmax(axis=1).str.replace('score_', '')
auto_labels['best_functional'] = cluster_score_mean[functional_cols].idxmax(axis=1).str.replace('score_', '')

# 组合命名: Identity_Immunoreg(+Functional 如果 functional score 显著高)
auto_labels['suggested_name'] = (
    auto_labels['best_identity'] + '_' + auto_labels['best_immunoreg']
)

# 如果某个 functional score 显著高于 identity/immunoreg, 追加到名称
for cl in auto_labels.index:
    func_top = auto_labels.loc[cl, 'best_functional']
    func_score = cluster_score_mean.loc[cl, f'score_{func_top}']
    id_score = cluster_score_mean.loc[cl, identity_cols].max()
    im_score = cluster_score_mean.loc[cl, immunoreg_cols].max()
    if func_score > max(id_score, im_score):
        auto_labels.loc[cl, 'suggested_name'] = func_top + '_dominant'

# 附加 cluster 大小
auto_labels.insert(0, 'n_cells', adata_mye.obs['leiden_sub'].value_counts().sort_index())

# 显示 identity / immunoreg / functional 的关键分数便于核对
key_cols = ['n_cells', 'suggested_name',
            'best_identity', 'best_immunoreg', 'best_functional']
auto_labels_display = auto_labels[key_cols].join(cluster_score_mean).round(3)
auto_labels_display.to_csv('write/4_Sub_clustering/myeloid_auto_annotation_helper.csv')
auto_labels_display


In [ ]:
# 各 cluster 的 cell_type 组成
ct_comp = pd.crosstab(adata_mye.obs['leiden_sub'], adata_mye.obs['cell_type'], normalize='index')
ct_comp = ct_comp.round(3)

# 加上 cluster size
ct_comp.insert(0, 'n_cells', adata_mye.obs['leiden_sub'].value_counts().sort_index())
ct_comp

In [ ]:
# === 严格 marker-based 注释: 用 scoring 结果的三组 best-hit 组合命名 ===
# 命名格式: Identity_ImmunoregProgram
# 若 functional marker 压倒性高表达 (G2-M/G1-S/Hypoxia/HS-UPR), 则用 Functional_dominant

mye_annot = {}
for cl in auto_labels.index:
    cl_str = str(cl)
    name = auto_labels.loc[cl, 'suggested_name']
    n = auto_labels.loc[cl, 'n_cells']
    mye_annot[cl_str] = name

# --- 手动修正: 基于 cell_type 组成核验 ---
mye_annot['0'] = 'Macrophage_Mg-Inflammatory'
mye_annot['5'] = 'Macrophage_Scavenger-IS'
mye_annot['9'] = 'Macrophage_Scavenger-IS'

# 打印注释表供核对
print("=== Marker-based annotation (with manual corrections) ===")
for cl_str in sorted(mye_annot.keys(), key=lambda x: int(x) if x.isdigit() else x):
    name = mye_annot[cl_str]
    n = auto_labels.loc[cl_str, 'n_cells'] if cl_str in auto_labels.index else '?'
    bi = auto_labels.loc[cl_str, 'best_identity'] if cl_str in auto_labels.index else '?'
    bm = auto_labels.loc[cl_str, 'best_immunoreg'] if cl_str in auto_labels.index else '?'
    bf = auto_labels.loc[cl_str, 'best_functional'] if cl_str in auto_labels.index else '?'
    note = ''
    if cl_str == '0': note = ' <- corrected: 98% Macro, cDC markers non-specific'
    elif cl_str == '5': note = ' <- corrected: 97% Macro, MG score too weak'
    elif cl_str == '9': note = ' <- corrected: G2-M score~0, not true cycling'
    print(f"  cl{cl_str}: {n:>5} cells → {name:<40s} (id={bi}, im={bm}, func={bf}){note}")

adata_mye.obs['myeloid_subtype'] = (
    adata_mye.obs['leiden_sub'].astype(str).map(mye_annot).astype('category')
)

# UMAP 展示注释结果
sc.pl.umap(adata_mye, color='myeloid_subtype', legend_loc='on data', legend_fontsize=6,
           title='Myeloid subtypes (marker-based)', save='_myeloid_annotated.png')
print('\n')
print(adata_mye.obs['myeloid_subtype'].value_counts())

In [ ]:
# Skip unified mapping CSV, use marker-based names directly
adata_mye.obs['myeloid_subtype_original'] = adata_mye.obs['myeloid_subtype'].astype(str).astype('category')
adata_mye.obs['myeloid_program_l1'] = adata_mye.obs['myeloid_subtype'].astype(str).astype('category')
adata_mye.obs['myeloid_program_l2'] = adata_mye.obs['myeloid_subtype'].astype(str).astype('category')
adata_mye.obs['myeloid_lineage_unified'] = adata_mye.obs['myeloid_subtype'].astype(str).astype('category')
adata_mye.obs['is_clean_myeloid'] = True

position_map = {'Peritumoral': 'Peripheral_like', 'Tumor': 'Tumor_like'}
tissue_labels = adata_mye.obs['tissue_type'].astype(str)
adata_mye.obs['tissue_position_unified'] = tissue_labels.map(position_map).fillna(tissue_labels).astype('category')

print('Marker-based subtype -> program_l1 (no CSV):')
print(pd.crosstab(adata_mye.obs['myeloid_subtype_original'], adata_mye.obs['myeloid_program_l1']))


In [ ]:
# Compare original subtypes and unified programs across Peritumoral vs Tumor
for label_col, prefix in [
    ('myeloid_subtype_original', 'myeloid_subtype'),
    ('myeloid_program_l1', 'myeloid_program_l1'),
]:
    count = pd.crosstab(adata_mye.obs[label_col], adata_mye.obs['tissue_position_unified'])
    frac_within_label = count.div(count.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
    frac_within_tissue = count.div(count.sum(axis=0).replace(0, np.nan), axis=1).fillna(0)

    count.to_csv(f'write/4_Sub_clustering/{prefix}_by_tissue_position_count.csv')
    frac_within_label.round(3).to_csv(f'write/4_Sub_clustering/{prefix}_by_tissue_position_fraction_within_label.csv')
    frac_within_tissue.round(3).to_csv(f'write/4_Sub_clustering/{prefix}_tissue_composition.csv')

    display_df = frac_within_label.round(3).sort_values('Tumor_like' if 'Tumor_like' in frac_within_label.columns else frac_within_label.columns[-1], ascending=False)
    print(f'\n{prefix}: fraction within label')
    print(display_df.to_string())


In [ ]:
import os
os.makedirs('write/4_Sub_clustering', exist_ok=True)

# 保存 myeloid subset（含 myeloid_subtype 注释）
adata_mye.write('write/4_Sub_clustering/adata_scMyeloid_annotated.h5ad')
print(f'✓ saved {adata_mye.n_obs} myeloid cells → write/4_Sub_clustering/adata_scMyeloid_annotated.h5ad')
print(adata_mye.obs['myeloid_subtype'].value_counts())

## 2. T/NK 亚群分析

In [ ]:
adata_T = adata[adata.obs['cell_type'] == 'NK/T'].copy()
print(f"NK/T: {adata_T.n_obs} cells")

sc.pp.highly_variable_genes(adata_T, n_top_genes=2000)
sc.tl.pca(adata_T, n_comps=30, use_highly_variable=True)
sce.pp.harmony_integrate(adata_T, key='sample', basis='X_pca', adjusted_basis='X_pca_harmony')
sc.pp.neighbors(adata_T, use_rep='X_pca_harmony', n_pcs=30)
sc.tl.umap(adata_T, min_dist=0.3)
sc.tl.leiden(adata_T, key_added='leiden_sub', resolution=0.5)
sc.pl.umap(adata_T, color=['cell_type', 'leiden_sub'], ncols=2, legend_loc='on data',
           save='_TNK_overview.png')

In [ ]:
# T/NK marker dotplot
tnk_markers = {
    'Pan-T': ['CD3D', 'CD3E', 'CD2'],
    'CD8 effector': ['CD8A', 'CD8B', 'GZMA', 'GZMB', 'PRF1', 'NKG7'],
    'CD8 exhausted': ['PDCD1', 'LAG3', 'HAVCR2', 'TIGIT', 'ENTPD1'],
    'CD4 Treg': ['CD4', 'FOXP3', 'IL2RA', 'CTLA4', 'IKZF2'],
    'CD4 Th': ['CD4', 'CCR6', 'RORA', 'ICOS'],
    'Naive/CM': ['CCR7', 'SELL', 'TCF7', 'LEF1'],
    'NK': ['NCAM1', 'KLRD1', 'KLRF1', 'FCGR3A', 'CD160'],
    'gdT': ['TRDC', 'TRGC1', 'TRGC2'],
}

sc.tl.dendrogram(adata_T, groupby='leiden_sub')
sc.pl.dotplot(adata_T, tnk_markers, groupby='leiden_sub', dendrogram=True,
              save='_TNK_markers.png')

In [ ]:
sc.tl.rank_genes_groups(adata_T, groupby='leiden_sub', method='wilcoxon', n_genes=10)
sc.pl.rank_genes_groups_dotplot(adata_T, n_genes=5, save='_TNK_deg.png')

In [ ]:
# T/NK 注释 (Harmony 后, 10 clusters)
tnk_annot = {
    '0': 'CD8 effector',      # CD3D++, GZMA/GZMB/PRF1/NKG7++, CCL5/GZMK/IFITM2
    '1': 'CD4 Th',            # CD4+, RORA++, CCR6+, BCL2/TOX → Th17-like
    '2': 'CD4 memory',        # CD3D+, SKAP1+, PIP4K2A, moderate markers
    '3': 'Myeloid contamination',  # FTL/FTH1/C1QB/APOE → myeloid markers, no Pan-T
    '4': 'CD8 pre-exhausted', # CD3D++, CD8+, PDCD1 moderate, IL32, ribosomal high
    '5': 'T-Tumor doublet',   # NLGN1/NRXN1/SOX6 neuronal → low Pan-T
    '6': 'Low-quality T',     # C1QB/APOE + ribosomal → Myeloid contamination
    '7': 'NK',                # GNLY/CTSW/PRF1/KLRD1/KLRF1++ → NK markers
    '8': 'Cycling T',         # MKI67/HMGB2/TUBA1B/NUSAP1 → proliferation
    '9': 'T-Tumor doublet',   # LSAMP/MIR99AHG/PLXDC2 neuronal → low Pan-T
}

adata_T.obs['tnk_subtype'] = adata_T.obs['leiden_sub'].map(tnk_annot).astype('category')

sc.pl.umap(adata_T, color='tnk_subtype', legend_loc='on data', legend_fontsize=7,
           title='T/NK subtypes', save='_TNK_annotated.png')
adata_T.obs['tnk_subtype'].value_counts()

## 3. Tumor 亚群分析

Tumor + Prol.tumor + NPC-like Tumor + OPC-like Tumor

In [ ]:
tumor_types = ['Tumor']
adata_tum = adata[adata.obs['cell_type'].isin(tumor_types)].copy()
print(f"Tumor compartment: {adata_tum.n_obs} cells")
adata_tum.obs['cell_type'].value_counts()

In [ ]:
sc.pp.highly_variable_genes(adata_tum, n_top_genes=3000)
sc.tl.pca(adata_tum, n_comps=50, use_highly_variable=True)

# Harmony 整合 sample batch effect
import scanpy.external as sce
sce.pp.harmony_integrate(adata_tum, key='sample', basis='X_pca', adjusted_basis='X_pca_harmony')

sc.pp.neighbors(adata_tum, n_pcs=30, use_rep='X_pca_harmony')
sc.tl.umap(adata_tum, min_dist=0.3)
sc.tl.leiden(adata_tum, key_added='leiden_sub', resolution=0.5)

sc.pl.umap(adata_tum, color=['sample', 'leiden_sub'], ncols=2, legend_loc='on data',
           legend_fontsize=5, save='_tumor_overview.png')

In [ ]:
# GBM 肿瘤细胞状态 markers (Neftel et al. 2019)
tumor_markers = {
    "MES1":  ["VIM", "LDHA", "ENO2", "ANXA1", "COL1A2", "TNC", "ZFP36L1", "CYR61"],
    "MES2":  ["HILPDA", "DDIT3", "HSPB1", "GPR37L1", "COL11A1", "PDLIM4"],
    "AC":    ["S100B", "GFAP", "SLC1A3", "MLC1", "HOPX", "ATP1A2", "CHL1", "CPNE1"],
    "OPC":   ["OLIG1", "OMG", "PLP1", "PLLP", "TNR", "ALCAM", "SOX10", "BEX4"],
    "NPC1":  ["SOX4", "SOX11", "DCX", "BEX1", "LSM3", "GAS5", "PLAT"],
    "NPC2":  ["STMN1", "STMN2", "STMN4", "DLX5-AS1", "DLX6-AS1", "GAD1"],
}

# 过滤不在 var_names 中的基因
available = set(adata_tum.var_names)
tumor_markers_filt = {k: [g for g in v if g in available] for k, v in tumor_markers.items()}
missing = {k: [g for g in v if g not in available] for k, v in tumor_markers.items()}
missing = {k: v for k, v in missing.items() if v}
if missing:
    print("缺失基因:", missing)

sc.tl.dendrogram(adata_tum, groupby='leiden_sub')
sc.pl.dotplot(adata_tum, tumor_markers_filt, groupby='leiden_sub', dendrogram=True,
              save='_tumor_state_markers.png')

In [ ]:
sc.tl.rank_genes_groups(adata_tum, groupby='leiden_sub', method='wilcoxon', n_genes=10)
sc.pl.rank_genes_groups_dotplot(adata_tum, n_genes=5, save='_tumor_deg.png')

In [ ]:
# Per-cell scoring: 用 Neftel 6 模块分数取 max，替代 cluster-based 注释
neftel_modules = {
    "MES1":  ["VIM", "LDHA", "ENO2", "ANXA1", "COL1A2", "TNC", "ZFP36L1"],
    "MES2":  ["HILPDA", "DDIT3", "HSPB1", "GPR37L1", "COL11A1", "PDLIM4"],
    "AC":    ["S100B", "GFAP", "SLC1A3", "MLC1", "HOPX", "ATP1A2", "CHL1", "CPNE1"],
    "OPC":   ["OLIG1", "OMG", "PLP1", "PLLP", "TNR", "ALCAM", "SOX10", "BEX4"],
    "NPC1":  ["SOX4", "SOX11", "DCX", "BEX1", "LSM3", "PLAT"],
    "NPC2":  ["STMN1", "STMN2", "STMN4", "DLX6-AS1", "GAD1"],
}

# 打分
available = set(adata_tum.var_names)
for mod, genes in neftel_modules.items():
    genes_use = [g for g in genes if g in available]
    sc.tl.score_genes(adata_tum, gene_list=genes_use, score_name=f'Neftel_{mod}', use_raw=False)
    print(f"{mod}: {len(genes_use)}/{len(genes)} genes")

# Cluster-level majority voting: 每个 leiden cluster 取平均分最高的模块
score_cols = [f'Neftel_{m}' for m in neftel_modules]
cluster_mean = adata_tum.obs.groupby('leiden_sub')[score_cols].mean()
cluster_state = cluster_mean.idxmax(axis=1).str.replace('Neftel_', '')

# Cycling: cluster 内 MKI67 均值 > 阈值
cycling_genes = ['MKI67', 'TOP2A', 'CENPF']
cycling_use = [g for g in cycling_genes if g in available]
sc.tl.score_genes(adata_tum, gene_list=cycling_use, score_name='Cycling_score', use_raw=False)
cluster_cycling = adata_tum.obs.groupby('leiden_sub')['Cycling_score'].mean()
cycling_clusters = cluster_cycling[cluster_cycling > 0.8].index
cluster_state[cycling_clusters] = 'Cycling'

# 打印 cluster → state 映射
for cl in cluster_state.index:
    scores = cluster_mean.loc[cl]
    top = scores.idxmax().replace('Neftel_', '')
    n = (adata_tum.obs['leiden_sub'] == cl).sum()
    cyc = '← Cycling' if cl in cycling_clusters else ''
    print(f"  cl{cl}: {n:>5} cells → {cluster_state[cl]:8s} (top={top}, score={scores.max():.2f}) {cyc}")

adata_tum.obs['tumor_subtype'] = adata_tum.obs['leiden_sub'].map(cluster_state).astype('category')
print("\n--- Cluster-level Neftel assignment ---")
print(adata_tum.obs['tumor_subtype'].value_counts())

In [ ]:
# 简洁 UMAP: 7 类 (6 核心态 + Cycling)
state_colors = {
    'AC': '#1f77b4', 'OPC': '#2ca02c', 'MES1': '#d62728',
    'MES2': '#ff7f0e', 'NPC1': '#9467bd', 'NPC2': '#8c564b',
    'Cycling': '#e377c2',
}
cats = adata_tum.obs['tumor_subtype'].cat.categories
palette = [state_colors.get(c, '#999999') for c in cats]

fig, ax = plt.subplots(figsize=(7, 6))
sc.pl.umap(adata_tum, color='tumor_subtype', legend_loc='right margin',
           palette=palette, title='Tumor states (Neftel)', ax=ax, show=False)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/umap_tumor_neftel_scoring.png', dpi=150, bbox_inches='tight')
plt.show()

# 6 个模块分数热图
sc.pl.umap(adata_tum, color=[f'Neftel_{m}' for m in ['MES1', 'MES2', 'AC', 'OPC', 'NPC1', 'NPC2']],
           ncols=3, vmin=-0.5, vmax=1.5, save='_tumor_neftel_scores.png')

## 4. 基质细胞亚群分析

Endothelial + Pericyte

In [ ]:
stromal_types = ['Endothelial', 'Pericyte']
adata_str = adata[adata.obs['cell_type'].isin(stromal_types)].copy()
print(f"Stromal: {adata_str.n_obs} cells")

sc.pp.highly_variable_genes(adata_str, n_top_genes=2000)
sc.tl.pca(adata_str, n_comps=30, use_highly_variable=True)
sce.pp.harmony_integrate(adata_str, key='sample', basis='X_pca', adjusted_basis='X_pca_harmony')
sc.pp.neighbors(adata_str, use_rep='X_pca_harmony', n_pcs=30)
sc.tl.umap(adata_str, min_dist=0.3)
sc.tl.leiden(adata_str, key_added='leiden_sub', resolution=0.3)
sc.pl.umap(adata_str, color=['cell_type', 'leiden_sub'], ncols=2, legend_loc='on data',
           save='_stromal_overview.png')

In [ ]:
stromal_markers = {
    'Endothelial': ['PECAM1', 'CDH5', 'VWF', 'CLDN5'],
    'Tip cell': ['ESM1', 'CXCR4', 'APLN', 'PGF'],
    'Stalk cell': ['DLL4', 'NOTCH1', 'HEY1'],
    'Lymphatic': ['LYVE1', 'PDPN', 'PROX1'],
    'Pericyte': ['PDGFRB', 'RGS5', 'ACTA2', 'NOTCH3'],
    'SMC': ['MYH11', 'TAGLN', 'CNN1'],
}

sc.tl.dendrogram(adata_str, groupby='leiden_sub')
sc.pl.dotplot(adata_str, stromal_markers, groupby='leiden_sub', dendrogram=True,
              save='_stromal_markers.png')

In [ ]:
sc.tl.rank_genes_groups(adata_str, groupby='leiden_sub', method='wilcoxon', n_genes=10)
sc.pl.rank_genes_groups_dotplot(adata_str, n_genes=5, save='_stromal_deg.png')

In [ ]:
# 各 cluster 的 cell_type 组成
ct_str = pd.crosstab(adata_str.obs['leiden_sub'], adata_str.obs['cell_type'], normalize='index').round(3)
ct_str.insert(0, 'n', adata_str.obs['leiden_sub'].value_counts().sort_index())
print(ct_str.to_string())

In [ ]:
# Stromal 亚型注释 (Harmony 后, 9 clusters)
stromal_annot = {
    '0': 'Activated EC',       # 1,597 | CCL3/FTL4/CCL4L2 | 92% EC
    '1': 'Venous EC',          # 698 | MECOM/PTPRM/CLDN5+ | 100% EC
    '2': 'Mural cell',         # 477 | COL1A1/LGALS1/PECAM1 | 83% Peri
    '3': 'CNS Pericyte',       # 465 | MIR99AHG/LSAMP/ZBTB20/PTPRZ1 | 67% Peri
    '4': 'Classic Pericyte',   # 398 | RGS5/NDUFA4L2/DCN/CALD1/PDGFRB++ | 68% Peri
    '5': 'Arterial EC',        # 379 | PTPRG/ST6GALNAC3/DOCK9 | 100% EC
    '6': 'Tip cell',           # 278 | SPP1/ANGPT2/SPARC/COL4A1 | 100% EC
    '7': 'BBB-like EC',        # 273 | CAVIN2/HSPG2/ANGPT2/CLDN5+++ highest | 100% EC
    '8': 'Doublet (Ig+)',      # 65 | IGLC2/IGHG3/BGN → Ig chains | 99% EC
}

adata_str.obs['stromal_subtype'] = adata_str.obs['leiden_sub'].map(stromal_annot).astype('category')

sc.pl.umap(adata_str, color='stromal_subtype', legend_loc='on data', legend_fontsize=7,
           title='Stromal subtypes', save='_stromal_annotated.png')
adata_str.obs['stromal_subtype'].value_counts()

## 5. 整合所有亚型注释

In [ ]:
# 将 4 个 compartment 的亚型注释合并到主 adata
subtype_map = {}
for sub_adata, col in [
    (adata_mye, 'myeloid_subtype'),
    (adata_T, 'tnk_subtype'),
    (adata_tum, 'tumor_subtype'),
    (adata_str, 'stromal_subtype'),
]:
    subtype_map.update(sub_adata.obs[col].to_dict())

adata.obs['Minor_cluster'] = adata.obs.index.map(subtype_map)

# Myeloid cells also get unified program labels; non-myeloid cells keep their Minor_cluster label.
myeloid_program_map = adata_mye.obs['myeloid_program_l1'].to_dict()
adata.obs['Minor_cluster_unified'] = adata.obs.index.map(myeloid_program_map)
adata.obs['Minor_cluster_unified'] = adata.obs['Minor_cluster_unified'].fillna(adata.obs['Minor_cluster'])

clean_map = adata_mye.obs['is_clean_myeloid'].to_dict()
adata.obs['is_clean_myeloid'] = adata.obs.index.map(clean_map).fillna(False).astype(bool)

print(f"已注释: {adata.obs['Minor_cluster'].notna().sum()} / {adata.n_obs}")
print(f"未注释: {adata.obs['Minor_cluster'].isna().sum()}")
print(adata.obs['Minor_cluster'].value_counts())
print('\nUnified myeloid programs:')
print(adata.obs['Minor_cluster_unified'].value_counts())


In [ ]:
# 全景 UMAP
sc.pl.umap(
    adata,
    color=['Minor_cluster', 'Minor_cluster_unified'],
    legend_loc='right margin',
    legend_fontsize=5,
    title=['All subtypes', 'All subtypes with unified myeloid programs'],
    save='_all_subtypes.png',
)

# 保存
adata.write('adata2.h5ad')
print('adata2.h5ad 已保存, 包含 Minor_cluster 和 Minor_cluster_unified 列')


In [ ]:
from pathlib import Path

old_path = Path('adata2.h5ad')
new_path = Path('write/0_integrate_two_datasets/adata2.h5ad')

adata_old = sc.read_h5ad(old_path)
adata_new = sc.read_h5ad(new_path)

cell_type_compare = pd.concat(
    [
        adata_old.obs['cell_type'].value_counts().rename('old_root'),
        adata_new.obs['cell_type'].value_counts().rename('new_integrated'),
    ],
    axis=1,
).fillna(0).astype(int)
cell_type_compare['delta'] = cell_type_compare['new_integrated'] - cell_type_compare['old_root']

myeloid_types = ['Macrophage', 'Microglia', 'Monocyte', 'Neutrophil']
myeloid_compare = pd.DataFrame(
    {
        'dataset': ['old_root', 'new_integrated'],
        'myeloid_n': [
            int(adata_old.obs['cell_type'].isin(myeloid_types).sum()),
            int(adata_new.obs['cell_type'].isin(myeloid_types).sum()),
        ],
        'total_n': [adata_old.n_obs, adata_new.n_obs],
    }
).set_index('dataset')
myeloid_compare['myeloid_fraction'] = (myeloid_compare['myeloid_n'] / myeloid_compare['total_n']).round(4)

print('Cell type comparison:')
print(cell_type_compare.sort_values('delta', ascending=False).to_string())
print('\nMyeloid summary:')
print(myeloid_compare.to_string())

## Marker-based tissue composition

按 tissue 统计每个 marker-based myeloid subtype 的组成比例。这里不使用 program_l1 或 unified mapping。


In [ ]:
# Marker-based tissue composition: fraction of each subtype within each tissue
subtype_col = 'myeloid_subtype'
tissue_col = 'tissue_position_unified'

marker_tissue_count = pd.crosstab(
    adata_mye.obs[tissue_col].astype(str),
    adata_mye.obs[subtype_col].astype(str),
)
marker_tissue_comp = marker_tissue_count.div(marker_tissue_count.sum(axis=1), axis=0).round(4)

marker_tissue_count.to_csv('write/4_Sub_clustering/myeloid_marker_based_tissue_count.csv')
marker_tissue_comp.to_csv('write/4_Sub_clustering/myeloid_marker_based_tissue_composition.csv')

print('Marker-based myeloid tissue composition:')
display(marker_tissue_comp)
print('Saved: write/4_Sub_clustering/myeloid_marker_based_tissue_count.csv')
print('Saved: write/4_Sub_clustering/myeloid_marker_based_tissue_composition.csv')

